## 课程实验

# D01：连接 Doris，查询第一批订单

Data Warehousing with Apache Doris · Level 1

[讲义](course.md) · [课程入口](../README.md)


## 实验范围

使用独立课程数据库；本 Lab 仅重建 d01_orders。运行前阅读 environments/README.md。目标是验证连接与正确性，不是测量性能。


In [ ]:
from dw_course import WarehouseLab
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows

lab = WarehouseLab()
print("Course database:", lab.database)



## 1. 记录真实环境

VERSION() 可能是协议兼容版本。对照 FE/BE 的版本与存活信息，不能将开发构建当作正式 4.1.3。


In [ ]:
print(lab.query("SELECT VERSION(), @@version_comment"))
print(lab.query("SHOW FRONTENDS"))
print(lab.query("SHOW BACKENDS"))


## 2. 建表并写入

先使用给定表结构；D03 再讨论为什么选择该模型。


In [ ]:
lab.execute("DROP TABLE IF EXISTS d01_orders")
ddl = order_ddl("d01_orders")
print(ddl)
lab.execute(ddl)
lab.insert("d01_orders", ORDER_COLUMNS, order_rows(fixture("orders.json")))


## 3. 用业务结果验收

结果必须为 10 笔订单、总金额 1400.00。明细按订单号排序，逐行观察状态和版本。


In [ ]:
print(lab.query("SELECT order_id, status, order_amount FROM d01_orders ORDER BY order_id"))
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM d01_orders"), [(10, "1400.00")])
lab.close()


## 完成与排查

连接失败先核对 FE 查询端口和认证；写入失败检查 BE Alive 与副本配置。不要为重试而删除整个数据库。下一单元观察这批数据在 Doris 内的组织方式。
